# LangWatch Async-Native Experiment Loop with Google ADK

This notebook demonstrates `experiment.aloop` + `experiment.asubmit` — the async sibling of the threading-based `loop` / `submit` pair.

**Why async?** Google ADK's `InMemoryRunner` binds a gRPC channel to the event loop it was created on. Under the threading path each worker runs its coroutine via `asyncio.run(...)`, which spins up a *new* loop per thread and breaks that binding (`"Future attached to a different loop"`). The async-native path keeps every item on the caller's event loop, so singletons and factories that open expensive connections once stay usable across items.

**Requirements**:
```bash
pip install langwatch google-adk openinference-instrumentation-google-adk
```
And set `LANGWATCH_API_KEY` + `GOOGLE_API_KEY` in your environment.

> **Setup:** the ADK runtime is in a separate `adk-examples` group (its pydantic pin conflicts with Chainlit in the main `examples` group). Install with `make install-adk-examples` before running this notebook.


In [1]:
import langwatch
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

langwatch.login()
langwatch.setup(instrumentors=[GoogleADKInstrumentor()])

LangWatch API key is already set, if you want to login again, please call as langwatch.login(relogin=True)
2026-04-17 10:21:26,625 - langwatch.client - INFO - Registering atexit handler to flush tracer provider on exit


/Users/rchaves/Projects/langwatch-async-experiment-parallelism/python-sdk/.venv/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## Create the agent and a shared runner

The runner is created **once**, on this notebook's event loop. Under the threading path, sharing it across concurrent items would raise `"Future attached to a different loop"`. With `aloop` / `asubmit`, every item runs on this same loop so the runner stays valid.

In [2]:
import os
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types


def get_weather(city: str) -> dict:
    forecasts = {
        "new york": "Sunny, 25C",
        "london": "Cloudy, 18C",
        "tokyo": "Rainy, 22C",
        "berlin": "Windy, 16C",
        "sao paulo": "Humid, 27C",
    }
    city = city.lower().strip()
    if city in forecasts:
        return {"status": "ok", "report": forecasts[city]}
    return {"status": "error", "message": f"no forecast for {city!r}"}


# Two agents, one per model, sharing the same tool and instruction so the
# comparison is apples-to-apples. Both runners stay alive for the whole
# experiment — this is exactly the loop-bound-singleton case that breaks
# on the threading path and is the reason we built `aloop` / `asubmit`.
AGENT_TARGETS = [
    # Both IDs are covered by langwatch's pricing registry, so cost
    # folds cleanly per target after each run.
    ("gemini-2.5-flash", "gemini-2.5-flash"),
    ("gemini-2.5-flash-lite", "gemini-2.5-flash-lite"),
]

agents = {
    name: Agent(
        name=f"weather_agent_{name.replace('-', '_').replace('.', '_')}",
        model=model_id,
        description="Replies with a short weather report when asked about a city.",
        instruction="Use get_weather, then answer in one short sentence.",
        tools=[get_weather],
    )
    for name, model_id in AGENT_TARGETS
}

runners = {
    name: InMemoryRunner(agent=agents[name], app_name=f"experiment-async-adk-{name}")
    for name, _ in AGENT_TARGETS
}


## Run the experiment with `aloop` + `asubmit` + multi-target

Each item asks both agents the same question inside its own
`experiment.target(...)` block — the platform records a per-target row
with its own trace, duration, cost, and `quality` score, so you can
compare the two models directly in the UI.

`concurrency=4` caps the number of in-flight tasks (bounded by
`asyncio.Semaphore`). Sync callables passed to `asubmit` would be
offloaded to a worker thread automatically, but here every task is
async — both runners stay usable on this loop.


In [3]:
from uuid import uuid4

cities = [
    "New York",
    "London",
    "Tokyo",
    "Berlin",
    "Sao Paulo",
    "New York",
    "London",
    "Tokyo",
    "Berlin",
    "Sao Paulo",
]

experiment = langwatch.experiment.init("async-adk-multi-target")

# Fresh prefix per cell execution so session IDs don't collide with earlier
# runs (ADK's in-memory session service raises AlreadyExistsError on duplicates).
RUN_ID = uuid4().hex[:8]

EXPECTED = {
    "new york": "sunny",
    "london": "cloudy",
    "tokyo": "rainy",
    "berlin": "windy",
    "sao paulo": "humid",
}


async def _run_agent(target_name: str, city: str, item_index: int) -> str:
    runner = runners[target_name]
    session_id = f"run-{RUN_ID}-{target_name}-{item_index}"
    await runner.session_service.create_session(
        app_name=runner.app_name, user_id="user", session_id=session_id
    )
    # IMPORTANT: drain the whole event stream. Early-returning inside this
    # `async for` leaves the generator to be GC'd later in the event loop's
    # context (not this task's), which tips ADK's OTel wrapper into
    # "Failed to detach context" spam on every iteration.
    response = ""
    async for event in runner.run_async(
        user_id="user",
        session_id=session_id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=f"What is the weather in {city}?")],
        ),
    ):
        if not event.is_final_response():
            continue
        content = getattr(event, "content", None)
        parts = getattr(content, "parts", None) or []
        text = getattr(parts[0], "text", None) if parts else None
        if text:
            response = text.strip()
    return response


async def ask_all_targets(city: str, *, item_index: int) -> None:
    expected_hint = EXPECTED.get(city.lower().strip(), "")
    for target_name, model_id in AGENT_TARGETS:
        with experiment.target(target_name, {"model": model_id}):
            response = await _run_agent(target_name, city, item_index)
            experiment.log_response(response)
            # Cheap deterministic grader so every row carries a score —
            # the real advanced case in offline_evaluation.ipynb does the
            # same pattern with random.uniform; here we reward the agent
            # for mentioning the expected keyword.
            score = 1.0 if expected_hint and expected_hint in response.lower() else 0.5
            experiment.log(
                "quality",
                index=item_index,
                score=score,
                passed=score >= 1.0,
                data={"response": response, "expected_hint": expected_hint},
            )


item_index = 0
async for city in experiment.aloop(cities, concurrency=4):
    experiment.asubmit(ask_all_targets, city, item_index=item_index)
    item_index += 1


Follow the results at: https://app.langwatch.ai/inbox-narrator/experiments/async-adk-multi-target?runId=thick-mellow-marten


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

entry  \
target                index              
gemini-2.5-flash      0       New York   
                      1         London   
                      2          Tokyo   
                      3         Berlin   
                      4      Sao Paulo   
                      5       New York   
                      6         London   
                      7          Tokyo   
                      8         Berlin   
                      9      Sao Paulo   
gemini-2.5-flash-lite 0       New York   
                      1         London   
                      2          Tokyo   
                      3         Berlin   
                      4      Sao Paulo   
                      5       New York   
                      6         London   
                      7          Tokyo   
                      8         Berlin   
                      9      Sao Paulo   

                                                                        output  \
target                index                                                      
gemini-2.5-flash      0                 The weather in New York is Sunny, 25C.   
                      1      The weather in London is cloudy with a tempera...   
                      2      The weather in Tokyo is rainy with a temperatu...   
                      3      The weather in Berlin is windy with a temperat...   
                      4      The weather in Sao Paulo is humid with a tempe...   
                      5      The weather in New York is sunny with a temper...   
                      6      The weather in London is cloudy with a tempera...   
                      7                          It is rainy and 22C in Tokyo.   
                      8                The weather in Berlin is windy and 16C.   
                      9                The weather in Sao Paulo is humid, 27C.   
gemini-2.5-flash-lite 0      The weather in New York is Sunny with a temper...   
                      1               The weather in London is cloudy and 18C.   
                      2                                                          
                      3                                                          
                      4                                                          
                      5             The weather in New York is sunny and 25°C.   
                      6                                                          
                      7                                                          
                      8                                                          
                      9             The weather in Sao Paulo is humid and 27C.   

                                                     trace_id  duration_ms  \
target                index                                                  
gemini-2.5-flash      0      c7ce8d960a5f6af4d3a7cff416df531f         2536   
                      1      969d415dbb6570005fccd1a1e017d5eb         2365   
                      2      3b23ec9ef8867a62279542908522a02f         1460   
                      3      b4893f8b0a71dd54aaf98e2d892c04cd         1241   
                      4      b50f6b12bea704267b4e1ec3e0937589         1617   
                      5      58b22917d2a9d1a0debeec66de87284d         1635   
                      6      05c346a8fd6ae175a93300bf95f7aff3         1429   
                      7      00ffc31eb2da5b8652d6348c624afdee         1673   
                      8      3af9c943f3c89b3e04a2ec4466598411         1260   
                      9      b5f28e9ca38342b3e485dc265c776682         2805   
gemini-2.5-flash-lite 0      2dfa26c51f53d9fe9cf0c02de73cec37         1045   
                      1      78a4ca34b03f13ecfeceac7fc9f56bd3         1022   
                      2      a0a0ce595d0047ffcefc8cd6a2244cff         1075   
                      3      aa908806add55a188e7f8ca5625a4b88         1286   
                      4      c5a250c4a5f3a753d3b49a3b13


  View details: https://app.langwatch.ai/inbox-narrator/experiments/async-adk-multi-target?runId=thick-mellow-marten
  Explore with: `evaluation.results`

